In [1]:
# Cell 1: Environment Setup & Data Ingestion
import pandas as pd
import numpy as np

# Display configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

# Load uploaded CSV file
file_name = 'housePrice.csv'
df_raw = pd.read_csv(file_name)
df = df_raw.copy()

print(f"Dataset successfully loaded: {df.shape[0]} rows, {df.shape[1]} columns.")
df.head()

Dataset successfully loaded: 3479 rows, 8 columns.


,Area,Room,Parking,Warehouse,Elevator,Address,Price,Price(USD)
0,63,1,True,True,True,Shahran,1850000000.00,61666.67
1,60,1,True,True,True,Shahran,1850000000.00,61666.67
2,79,2,True,True,True,Pardis,550000000.00,18333.33
3,95,2,True,True,True,Shahrake Qods,902500000.00,30083.33
4,123,2,True,True,True,Shahrake Gharb,7000000000.00,233333.33


In [2]:
# Cell 2: Data Quality Diagnostic Report
quality_report = pd.DataFrame({
    'Data Type': df.dtypes,
    'Non-Null Count': df.notnull().sum(),
    'Null Count': df.isnull().sum(),
    'Null Pct (%)': (df.isnull().sum() / len(df) * 100).round(2),
    'Unique Values': df.nunique()
})

print("--- Data Quality Diagnostic Table ---")
display(quality_report)

raw_dupes = df.duplicated().sum()
print(f"\n[Audit] Exact Duplicate Rows Found: {raw_dupes}")

print("\n--- Summary Statistics (Raw Data) ---")
display(df.describe(include='all'))

--- Data Quality Diagnostic Table ---


,Data Type,Non-Null Count,Null Count,Null Pct (%),Unique Values
Area,object,3479,0,0.00,243
Room,int64,3479,0,0.00,6
Parking,bool,3479,0,0.00,2
Warehouse,bool,3479,0,0.00,2
Elevator,bool,3479,0,0.00,2
Address,object,3456,23,0.66,192
Price,float64,3479,0,0.00,934
Price(USD),float64,3479,0,0.00,932



[Audit] Exact Duplicate Rows Found: 208

--- Summary Statistics (Raw Data) ---


,Area,Room,Parking,Warehouse,Elevator,Address,Price,Price(USD)
count,3479,3479.00,3479,3479,3479,3456,3479.00,3479.00
unique,243,NaN,2,2,2,192,NaN,NaN
top,75,NaN,True,True,True,Punak,NaN,NaN
freq,111,NaN,2950,3182,2739,161,NaN,NaN
mean,NaN,2.08,NaN,NaN,NaN,NaN,5359022710.58,178634.09
std,NaN,0.76,NaN,NaN,NaN,NaN,8099934524.33,269997.82
min,NaN,0.00,NaN,NaN,NaN,NaN,3600000.00,120.00
25%,NaN,2.00,NaN,NaN,NaN,NaN,1418250000.00,47275.00
50%,NaN,2.00,NaN,NaN,NaN,NaN,2900000000.00,96666.67
75%,NaN,2.00,NaN,NaN,NaN,NaN,6000000000.00,200000.00


In [3]:
# Cell 3: Clean Area Strings and Boolean Attributes
# 1. Clean Area: remove comma separators, strip whitespace, cast to numeric
df['Area'] = df['Area'].astype(str).str.replace(',', '', regex=False).str.strip()
df['Area'] = pd.to_numeric(df['Area'], errors='coerce')

# 2. Standardise boolean indicator columns
for col in ['Parking', 'Warehouse', 'Elevator']:
    if col in df.columns:
        df[col] = df[col].astype(bool)

print("Standardisation complete. Current column dtypes:")
print(df.dtypes)

Standardisation complete. Current column dtypes:
Area            int64
Room            int64
Parking          bool
Warehouse        bool
Elevator         bool
Address        object
Price         float64
Price(USD)    float64
dtype: object


In [4]:
# Cell 4: Duplicate Detection & Removal
rows_before = len(df)
dupes_count = df.duplicated().sum()

df = df.drop_duplicates().reset_index(drop=True)

print(f"Duplicates identified and removed: {dupes_count}")
print(f"Row count updated: {rows_before} -> {len(df)}")

Duplicates identified and removed: 208
Row count updated: 3479 -> 3271


In [5]:
# Cell 5: Missing Value Imputation
# 1. Address imputation & string standardisation
address_missing = df['Address'].isnull().sum()
df['Address'] = df['Address'].fillna('Unknown').astype(str).str.strip().str.title()
print(f"Address: Filled {address_missing} missing values with 'Unknown' and title-cased.")

# 2. Check and handle any remaining numeric missing values
for col in ['Area', 'Room', 'Price', 'Price(USD)']:
    if df[col].isnull().sum() > 0:
        med_val = df[col].median()
        df[col] = df[col].fillna(med_val)
        print(f"{col}: Imputed missing values with median ({med_val}).")

print(f"\nRemaining Nulls Across Dataset: {df.isnull().sum().sum()}")

Address: Filled 23 missing values with 'Unknown' and title-cased.

Remaining Nulls Across Dataset: 0


In [6]:
# Cell 6: Outlier Detection and Capping
# 1. Remove impossible area data-entry errors (> 2,000 m²)
corrupted_area_mask = df['Area'] > 2000
print(f"Dropped {corrupted_area_mask.sum()} records with corrupted Area (> 2,000 m²).")
df = df[~corrupted_area_mask].reset_index(drop=True)

# 2. Calculate IQR bounds
def get_iqr_bounds(series):
    q25 = series.quantile(0.25)
    q75 = series.quantile(0.75)
    iqr = q75 - q25
    lower_bound = max(0, q25 - 1.5 * iqr)
    upper_bound = q75 + 1.5 * iqr
    return lower_bound, upper_bound

area_low, area_high = get_iqr_bounds(df['Area'])
price_low, price_high = get_iqr_bounds(df['Price'])

print(f"Area Statistical IQR Bounds:  [{area_low:.1f}, {area_high:.1f}] m²")
print(f"Price Statistical IQR Bounds: [{price_low:,.0f}, {price_high:,.0f}]")

# 3. 99th Percentile Capping (Winsorization) using numpy
area_cap = df['Area'].quantile(0.99)
price_cap = df['Price'].quantile(0.99)

df['Area_Capped'] = np.where(df['Area'] > area_cap, area_cap, df['Area'])
df['Price_Capped'] = np.where(df['Price'] > price_cap, price_cap, df['Price'])

print(f"Capping applied: Area capped at {area_cap:.1f} m², Price capped at {price_cap:,.0f}")

Dropped 5 records with corrupted Area (> 2,000 m²).
Area Statistical IQR Bounds:  [0.0, 197.5] m²
Price Statistical IQR Bounds: [0, 13,123,187,500]
Capping applied: Area capped at 400.0 m², Price capped at 40,700,000,000


In [7]:
# Cell 7: Final Type Casting
df['Area'] = df['Area'].astype(float)
df['Room'] = df['Room'].astype(int)
df['Parking'] = df['Parking'].astype(bool)
df['Warehouse'] = df['Warehouse'].astype(bool)
df['Elevator'] = df['Elevator'].astype(bool)
df['Address'] = df['Address'].astype('category')
df['Price'] = df['Price'].astype(float)
df['Price(USD)'] = df['Price(USD)'].astype(float)

print("Final Verified Column Dtypes:")
print(df.dtypes)

Final Verified Column Dtypes:
Area             float64
Room               int64
Parking             bool
Warehouse           bool
Elevator            bool
Address         category
Price            float64
Price(USD)       float64
Area_Capped      float64
Price_Capped     float64
dtype: object


In [8]:
# Cell 8: Before vs. After Audit Table
before_after_df = pd.DataFrame({
    'Metric': [
        'Total Rows',
        'Total Columns',
        'Duplicate Rows',
        'Missing Values (Address)',
        'Corrupted Area Format/Values',
        'Area Data Type',
        'Data Quality Status'
    ],
    'Before Cleaning': [
        df_raw.shape[0],
        df_raw.shape[1],
        df_raw.duplicated().sum(),
        df_raw['Address'].isnull().sum() if 'Address' in df_raw.columns else 0,
        (df_raw['Area'].astype(str).str.contains(',')).sum(),
        str(df_raw['Area'].dtype),
        'Dirty (Nulls, Duplicates, Malformed Strings)'
    ],
    'After Cleaning': [
        len(df),
        df.shape[1],
        df.duplicated().sum(),
        df['Address'].isnull().sum(),
        0,
        str(df['Area'].dtype),
        '100% Analysis-Ready'
    ]
})

display(before_after_df)

,Metric,Before Cleaning,After Cleaning
0,Total Rows,3479,3266
1,Total Columns,8,10
2,Duplicate Rows,208,0
3,Missing Values (Address),23,0
4,Corrupted Area Format/Values,6,0
5,Area Data Type,object,float64
6,Data Quality Status,"Dirty (Nulls, Duplicates, Malformed Strings)",100% Analysis-Ready


In [9]:
# Cell 9: Save Clean CSV
output_file = 'cleaned_housePrice.csv'
df.to_csv(output_file, index=False)
print(f"Successfully saved {len(df)} cleaned rows to '{output_file}'.")
df.head()

Successfully saved 3266 cleaned rows to 'cleaned_housePrice.csv'.


,Area,Room,Parking,Warehouse,Elevator,Address,Price,Price(USD),Area_Capped,Price_Capped
0,63.00,1,True,True,True,Shahran,1850000000.00,61666.67,63.00,1850000000.00
1,60.00,1,True,True,True,Shahran,1850000000.00,61666.67,60.00,1850000000.00
2,79.00,2,True,True,True,Pardis,550000000.00,18333.33,79.00,550000000.00
3,95.00,2,True,True,True,Shahrake Qods,902500000.00,30083.33,95.00,902500000.00
4,123.00,2,True,True,True,Shahrake Gharb,7000000000.00,233333.33,123.00,7000000000.00
